# Numerical Implementation of the Jump-Diffusion Model

The jump-diffusion framework presented here implements a discrete-time numerical version of the classical model, incorporating stochastic jumps into a step-by-step simulation of asset returns. This specific implementation provides a computationally efficient representation of price dynamics, explicitly calculating return factors to capture both baseline fluctuations and sudden market shocks.

## 1. Fundamental Assumptions

This numerical implementation relies on time discretization (Euler scheme) and adapts classical assumptions for direct computation on discrete returns:

* **Hybrid Discrete Dynamics:** Asset prices evolve through discrete return factors combining a deterministic drift, a uniform stochastic perturbation, and random jumps.
* **Bernoulli Jump Arrivals:** In each time interval $\Delta t$, the probability of a jump is approximated by $\lambda \Delta t$, simulated by comparing uniform random numbers to this intensity threshold.
* **Jump Size Distribution:** Jump magnitudes are drawn from a normal distribution and applied additively to the gross return of the single time step.
* **Uniform Innovation:** Unlike the standard continuous model, the baseline stochastic component is driven by a uniform distribution $\mathcal{U}(0,1)$, introducing a strictly non-negative shock to the returns.
* **Vectorization:** The market is simulated in parallel across multiple independent paths without the use of iterative loops, ensuring high computational efficiency.

## 2. Key Model Parameters

The script specification includes the following computational parameters:

1. $S_0$: initial asset price at time zero.
2. $\mu$: drift rate applied to the continuous component.
3. $\sigma$: volatility parameter governing baseline fluctuations.
4. $\lambda$: annual intensity (expected frequency) of the jumps.
5. $\mu_J$: mean of the jump amplitude applied to the return.
6. $\sigma_J$: standard deviation of the jump amplitude.
7. $\Delta t$: time step size, calculated as $T/N$.

## 3. Mathematical Formulation

The fundamental process for updating the price is governed by the discrete return factor $Q_t$ in each sub-interval:

$$
Q_t = 1 + \mu \Delta t + \sigma \sqrt{\Delta t} \, Z_t + J_t
$$

where:
* $Z_t \sim \mathcal{U}(0,1)$ is a standard uniform random variable;
* $J_t$ denotes the jump amplitude in the period $\Delta t$;
* $J_t = \sigma_J X + \mu_J$ (with $X \sim \mathcal{N}(0,1)$) if a jump occurs, otherwise $J_t = 0$.

The asset price at time $T$ is then calculated through the cumulative product:

$$
S_T = S_0 \prod_{i=1}^{N} Q_i
$$

This decomposition highlights how the multiplicative factor is the sum of three elements: the expected baseline return ($1 + \mu \Delta t$), the continuous volatility-dependent shock ($\sigma \sqrt{\Delta t} \, Z_t$), and the stochastic impact of the jump ($J_t$).

## 4. Path Construction

Under this numerical framework, matrix simulation allows generating the entire price surface in a vectorized manner. The simulation matrix $S$ is built by concatenating a vector of ones (representing the initial time $t=0$) with the factors $Q_t$:

$$
S = S_0 \cdot \text{cumprod}([1, Q_1, Q_2, \ldots, Q_N])
$$

This formula reveals that each final price trajectory is the result of compound interest applied to stochastic return factors, allowing the price evolution to be tracked step-by-step from the initial condition to the horizon $T$.

## 5. Interpretation and Use

This numerical setup offers specific practical advantages for software implementation:

* it guarantees **extremely high computational efficiency** in MATLAB by leveraging matrix logical evaluation (`U < intensity`);
* it captures the effect of **instantaneous price shocks** by combining linear returns and jumps into a single discrete random variable;
* it models a **positive asymmetric drift**, useful in scenarios where market innovations are assumed as constant bullish drivers (due to the use of the uniform distribution);
* it extends naturally to **massive Monte Carlo simulations**, serving as a basis for pricing path-dependent derivatives in the absence of closed-form solutions.

## 6. Limitations

Despite its efficiency, this specific numerical formulation has inherent structural constraints:

* **Non-standard drift:** The use of $\mathcal{U}(0,1)$ instead of a Brownian increment $\mathcal{N}(0,1)$ alters the expected value and the theoretical variance of the baseline process.
* **Risk of negative prices:** The additivity of a normal jump within a linear multiplicative factor can generate $Q_t < 0$ in the case of extreme jumps, pushing the price $S_t$ below zero.
* **Bernoulli approximation:** For large $\Delta t$ or extreme intensities $\lambda$, the jump probability $\lambda \Delta t$ could exceed 1, invalidating the simulation of arrivals.
* **Theoretical divergence:** The scheme deviates from the exact log-price solution (classical Merton), making direct calibration with real market parameters complex.

## 7. Extensions and Generalizations

Natural extensions of this script include:

* **Transition to log-returns:** Replacing the linear scheme with the exponential of the Brownian motion to ensure strict price positivity.
* **Gaussian innovation:** Replacing `rand` with `randn` to align the continuous noise with the theory of Wiener processes.
* **Exact Poisson processes:** Implementing the generation of exponential inter-arrival times instead of the Bernoulli discretization for jumps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# simulazione processo JD

# Fissiamo il seed per la riproducibilità (equivalente a rng in MATLAB)
np.random.seed(160524)

M = 10
T = 1
N = 252
dt = T / N

S0 = 90
mu = 0.05
sig = 0.20
lam = 30
mu_J = -0.02
sig_J = 0.22

# Generazione numeri casuali uniformi per l'arrivo dei salti
U = np.random.rand(M, N)
intensity = lam * dt

# Matrice booleana (True dove avviene un salto)
ind = U < intensity
N_jump = np.sum(ind)

# Ampiezza del salto (solo per gli eventi in cui ind è True)
# np.random.randn genera numeri da una normale standard
H_jump = sig_J * np.random.randn(N_jump) + mu_J

J = np.zeros((M, N))
J[ind] = H_jump

# Innovazione continua (uniforme, come nel tuo codice originale)
Z = np.random.rand(M, N)
Q = 1 + mu * dt + sig * np.sqrt(dt) * Z + J

# Vettore colonna di '1' per il tempo iniziale
ones_col = np.ones((M, 1))

# Concateniamo orizzontalmente la colonna di 1 con la matrice Q
Q_concat = np.hstack((ones_col, Q))

# Calcolo delle traiettorie con prodotto cumulativo lungo l'asse 1 (righe)
S = S0 * np.cumprod(Q_concat, axis=1)

# Plotting dei risultati
t = np.arange(0, T + dt, dt)  # Vettore dei tempi da 0 a T

plt.figure(figsize=(10, 6))
# Matplotlib richiede che S sia trasposta (S.T) per plottare correttamente le righe come traiettorie
plt.plot(t, S.T) 
plt.title("Simulazione Processo Jump-Diffusion (Implementazione Numerica)")
plt.xlabel("Tempo (Anni)")
plt.ylabel("Prezzo Asset")
plt.grid(True)
plt.show()